In [ ]:
# Week 1 - Applied Data Science with AI
# Project: E-Commerce Recommendation System
# Dataset: Ecommerce Customer Service Satisfaction

# Import libraries
import pandas as pd
from tabulate import tabulate

# Expand display
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# Load dataset (make sure the dataset is uploaded in Colab or working directory)
df = pd.read_csv("Customer_support_data.csv", encoding="utf-8")

# ---- Dataset Info ----
dataset_name = "Customer_support_data"
rows, cols = df.shape

# Table for dataset rows & columns
dataset_shape = pd.DataFrame({
    "Dataset Name": [dataset_name],
    "Total Rows": [rows],
    "Total Columns": [cols]
})

print("📌 Dataset Information:")
print(tabulate(dataset_shape, headers="keys", tablefmt="grid"))

# ---- Column Descriptions ----
# (Modify according to actual dataset columns)
column_info = {
    "CustomerID": "Unique identifier for each customer",
    "Gender": "Gender of the customer",
    "Age": "Age of the customer",
    "Country": "Country of residence",
    "ProductCategory": "Category of the purchased product",
    "Rating": "Customer rating (1–5)",
    "Review": "Customer review text/feedback"
}

print("\n📌 Column Descriptions:")
column_desc_table = pd.DataFrame(list(column_info.items()), columns=["Column Name", "Description"])
print(tabulate(column_desc_table, headers="keys", tablefmt="grid"))

# ---- Sample Data ----
print("\n📌 Dataset Sample (First 10 Rows):")
print(tabulate(df.head(10), headers="keys", tablefmt="grid"))

# ---- Missing Values ----
missing_values = df.isnull().sum().reset_index()
missing_values.columns = ["Column Name", "Missing Values"]

print("\n📌 Missing Values per Column:")
print(tabulate(missing_values, headers="keys", tablefmt="grid"))

# ---- Dataset Summary ----
info_data = {
    "Total Rows": [rows],
    "Total Columns": [cols],
    "Duplicate Rows": [df.duplicated().sum()],
    "Missing Cells": [df.isnull().sum().sum()],
    "Numeric Columns": [df.select_dtypes(include='number').shape[1]],
    "Categorical Columns": [df.select_dtypes(exclude='number').shape[1]]
}

print("\n📌 Dataset Summary:")
print(tabulate(pd.DataFrame(info_data), headers="keys", tablefmt="grid"))

# =========================================
# Week 2 - Data Cleaning (E-Commerce Project)
# Dataset: Customer_support_data.csv
# =========================================

import pandas as pd
import numpy as np
from tabulate import tabulate
from IPython.display import display

# ---- 1. Load Dataset ----
df_before = pd.read_csv("Customer_support_data.csv")   # Keep original for comparison
df = df_before.copy()  # Working copy

print("📌 Dataset Loaded Successfully!")

# ---- 2. Dataset Shape ----
print("\n📌 Dataset Shape (Before Cleaning):")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")

# ---- 3. Missing Values Count ----
print("\n📌 Missing Values Per Column (Before Cleaning):")
missing_values = df.isnull().sum().reset_index()
missing_values.columns = ["Column", "Missing Values"]
print(tabulate(missing_values, headers="keys", tablefmt="grid"))

# ---- 4. Handle Missing Values ----
missing_mask = df.isnull()   # Store NaN locations before filling

filling_strategy = []
for col in df.columns:
    if df[col].dtype in ["int64","float64"]:  # Numeric column
        mean_val = round(df[col].mean(skipna=True),2) if not df[col].dropna().empty else np.nan
        median_val = round(df[col].median(skipna=True),2) if not df[col].dropna().empty else np.nan
        mode_val = df[col].mode().iloc[0] if not df[col].mode().empty else np.nan
        strategy = "Median"
        df[col] = df[col].fillna(median_val)
    else:  # Categorical column
        mean_val, median_val = np.nan, np.nan
        mode_val = df[col].mode().iloc[0] if not df[col].mode().empty else "N/A"
        strategy = "Mode"
        df[col] = df[col].fillna(mode_val)
    
    filling_strategy.append([col, mean_val, median_val, mode_val, strategy])

# Print Filling Strategy Table
print("\n📌 Missing Value Filling Strategy (Mean / Median / Mode):")
print(tabulate(pd.DataFrame(filling_strategy, 
                            columns=["Column","Mean","Median","Mode","Used Strategy"]),
               headers="keys", tablefmt="grid", showindex=False))

# ---- 5. Remove Duplicates ----
before_dup = df.shape[0]
df = df.drop_duplicates()
after_dup = df.shape[0]
print(f"\n📌 Duplicate Rows Removed: {before_dup - after_dup}")







# ---- 6. Outlier Detection & Treatment with Explanation ----
print("\n📌 Quartile & Outlier Explanation (Numeric Columns):")

iqr_summary = []  # store results for summary table

for col in df.select_dtypes(include=["int64","float64"]).columns:
    Q1 = df[col].quantile(0.25)
    Q2 = df[col].quantile(0.50)  # Median
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    below_outliers = (df[col] < lower).sum()
    above_outliers = (df[col] > upper).sum()
    
    # Print explanation with conceptual meaning
    print(f"\n📊 Column: {col}")
    print(f"   Q1 (middle of lower half) = {Q1:.2f}")
    print(f"   Q2 (median / middle of dataset) = {Q2:.2f}")
    print(f"   Q3 (middle of upper half) = {Q3:.2f}")
    print(f"   IQR (Q3 - Q1) = {IQR:.2f}")
    print(f"   Lower Bound = Q1 - 1.5*IQR = {lower:.2f}")
    print(f"   Upper Bound = Q3 + 1.5*IQR = {upper:.2f}")
    
    # Added note explaining bounds
    print(f"   Note: Values >= {lower:.2f} and <= {upper:.2f} are considered normal;")
    print(f"         only values < {lower:.2f} or > {upper:.2f} are outliers.")
    
    print(f"   ➝ Outliers Found: {below_outliers} below, {above_outliers} above")
    
    # Store results for table
    iqr_summary.append([col, round(Q1,2), round(Q2,2), round(Q3,2), round(IQR,2),
                        round(lower,2), round(upper,2), below_outliers, above_outliers])
    
    # Cap the outliers
    df[col] = np.where(df[col] < lower, lower, df[col])
    df[col] = np.where(df[col] > upper, upper, df[col])

# Create summary table
iqr_table = pd.DataFrame(iqr_summary, 
                         columns=["Column", "Q1", "Q2 (Median)", "Q3", "IQR", 
                                  "Lower Bound", "Upper Bound", "Outliers Below", "Outliers Above"])

print("\n📌 IQR & Outlier Summary Table:")
print(tabulate(iqr_table, headers="keys", tablefmt="grid", showindex=False))

# ---- 7. Show First 20 Rows Before & After Cleaning ----
print("\n📌 First 20 Rows BEFORE Cleaning (Original Data with NaNs):")
display(df_before.head(20))

print("\n📌 First 20 Rows AFTER Cleaning (Missing Values Filled, Highlighted in Yellow):")

def highlight_filled(val, was_missing):
    return "background-color: #D97D55" if was_missing else ""

styled = df.head(20).style.apply(
    lambda s: [highlight_filled(v, was_missing) 
               for v, was_missing in zip(s, missing_mask[s.name].head(20))],
    axis=0
)
display(styled)

# ---- 8. Summary After Cleaning ----
print("\n📌 Dataset Shape (After Cleaning):")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")

print("\n📌 Missing Values Per Column (After Cleaning):")
missing_values_after = df.isnull().sum().reset_index()
missing_values_after.columns = ["Column", "Missing Values"]
print(tabulate(missing_values_after, headers="keys", tablefmt="grid"))

# ===============================
# Week 3 Assignment - E-commerce Customer Service Satisfaction (EDA)
# ===============================

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import textwrap

# -------------------------------
# 1. Load Dataset
# -------------------------------
df = pd.read_csv("Customer_support_data.csv")   # replace with your filename

# -------------------------------
# 2. Dataset Information as Table
# -------------------------------
info_table = pd.DataFrame({
    "Column Name": df.columns,
    "Non-Null Count": df.notnull().sum().values,
    "Missing Values": df.isnull().sum().values,
    "Data Type": df.dtypes.values
})

print("===== DATASET INFORMATION =====")
display(info_table)

# -------------------------------
# 3. Statistical Summaries (Numeric Features)
# -------------------------------
def mode_str(series):
    m = series.mode(dropna=True)
    return ", ".join(str(x) for x in m.tolist()) if len(m) > 0 else ""

def feature_summary(series, feature_name):
    return {
        "Feature": feature_name,
        "Count (Non-Null)": series.count(),
        "Missing": series.isna().sum(),
        "Mean": round(series.mean(), 2),
        "Median": round(series.median(), 2),
        "Mode": mode_str(series),
        "Std Dev": round(series.std(), 2),
        "Min": round(series.min(), 2),
        "Max": round(series.max(), 2)
    }

numeric_cols = ["Item_price", "connected_handling_time", "CSAT Score"]
summaries = []
for col in numeric_cols:
    if col in df.columns:
        summaries.append(feature_summary(df[col], col))

stats_df = pd.DataFrame(summaries)
print("\n===== NUMERIC FEATURE STATISTICS =====")
display(stats_df)

# -------------------------------
# 4. Distribution Plots
# -------------------------------
for col in numeric_cols:
    if col in df.columns:
        plt.figure(figsize=(7,4))
        sns.histplot(df[col].dropna(), kde=True, bins=30, color="skyblue")
        plt.title(f"Distribution of {col}")
        plt.xlabel(col)
        plt.ylabel("Count")
        plt.show()

# -------------------------------
# 5. Count Plots (Categorical Features)
# -------------------------------
cat_feats = ["channel_name", "category", "Sub-category", "Agent Shift", "Tenure Bucket"]

for col in cat_feats:
    if col in df.columns:
        plt.figure(figsize=(10,5))
        
        if col == "Sub-category":
            # Show top 15 subcategories only
            top_subs = df["Sub-category"].value_counts().nlargest(10).index
            sub_df = df[df["Sub-category"].isin(top_subs)]
            ax = sns.countplot(
                x="Sub-category",
                data=sub_df,
                order=top_subs,
                hue="Sub-category",
                legend=False,
                palette="Set2"
            )
            # Wrap long labels safely
            ax.set_xticks(ax.get_xticks())
            ax.set_xticklabels([textwrap.fill(lbl.get_text(), 12) for lbl in ax.get_xticklabels()])
        else:
            ax = sns.countplot(
                x=col,
                data=df,
                order=df[col].value_counts().index,
                hue=col,
                legend=False,
                palette="Set2"
            )
            ax.set_xticks(ax.get_xticks())
            ax.set_xticklabels([lbl.get_text() for lbl in ax.get_xticklabels()], rotation=45, ha="right")

        plt.title(f"Count of {col}")
        plt.tight_layout()
        plt.show()

# -------------------------------
# 6. Bar Plots (Avg CSAT Score by Categories)
# -------------------------------
if "channel_name" in df.columns and "CSAT Score" in df.columns:
    plt.figure(figsize=(7,4))
    sns.barplot(
        x="channel_name",
        y="CSAT Score",
        data=df,
        estimator="mean",
        hue="channel_name",
        legend=False,
        palette="Blues_d"
    )
    plt.title("Average CSAT Score by Channel")
    plt.xticks(rotation=30)
    plt.show()

if "category" in df.columns and "CSAT Score" in df.columns:
    top_cats = df["category"].value_counts().nlargest(10).index
    plt.figure(figsize=(10,5))
    sns.barplot(
        x="category",
        y="CSAT Score",
        data=df[df["category"].isin(top_cats)],
        estimator="mean",
        hue="category",
        legend=False,
        palette="Greens_d"
    )
    plt.title("Average CSAT Score by Category (Top 10)")
    plt.xticks(rotation=30)
    plt.show()

# -------------------------------
# 7. Boxplots (Numeric Features vs CSAT Score)
# -------------------------------

if "CSAT Score" in df.columns:
    for col in ["Item_price", "connected_handling_time"]:
        if col in df.columns:
            plt.figure(figsize=(7,4))
            sns.boxplot(
                x="CSAT Score",
                y=col,
                data=df,
                hue="CSAT Score",
                legend=False,
                palette="Set3"
            )
            plt.title(f"{col} Distribution by CSAT Score")
            plt.show()

# -------------------------------
# 8. Correlation Heatmap
# -------------------------------
plt.figure(figsize=(8,6))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap of Numeric Features")
plt.show()

# ===============================================================
# 📘 WEEK 4 - CORRELATION ANALYSIS (PROFESSIONAL FINAL VERSION)
# ===============================================================

# ------------------------------
# Step 1: Import Required Libraries
# ------------------------------
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from IPython.display import display, HTML

# ------------------------------
# Step 2: Load Dataset
# ------------------------------
file_path = "Customer_support_data.csv"  # Update your dataset path if needed
df = pd.read_csv(file_path)

print("✅ Dataset Loaded Successfully")

display(HTML("<h4>📊 Dataset Overview</h4>"))
display(HTML(f"<b>Shape:</b> {df.shape[0]} rows × {df.shape[1]} columns"))
display(HTML("<b>Column Names:</b>"))
display(pd.DataFrame({"Columns": df.columns.tolist()}))

display(HTML("<b>First 5 Rows of the Dataset:</b>"))
display(df.head())

# ------------------------------
# Step 3: Handle Missing Values
# ------------------------------
df.replace("?", np.nan, inplace=True)
df.fillna(df.median(numeric_only=True), inplace=True)
df.fillna(df.mode().iloc[0], inplace=True)

missing_after = df.isnull().sum().sum()
display(HTML(f"<h4>🧹 Missing Value Check:</h4><p>Total Missing Values After Cleaning: <b>{missing_after}</b></p>"))

# ------------------------------
# Step 4: Encode Categorical Columns
# ------------------------------
cat_cols = df.select_dtypes(include=["object"]).columns
if len(cat_cols) > 0:
    le = LabelEncoder()
    for col in cat_cols:
        df[col] = le.fit_transform(df[col].astype(str))
    display(HTML(f"<h4>🔠 Encoded {len(cat_cols)} Categorical Columns:</h4>"))
    display(pd.DataFrame({"Categorical Columns": cat_cols}))
else:
    display(HTML("<h4>🔠 No Categorical Columns Found</h4>"))

# ------------------------------
# Step 5: Detect Target Column
# ------------------------------
possible_targets = ["CSAT Score", "Satisfaction", "Customer_Satisfaction", "Rating", "rating"]
target_col = None

for col in df.columns:
    if col in possible_targets:
        target_col = col
        break

if target_col:
    display(HTML(f"<h4>🎯 Target Column Detected:</h4><p><b>{target_col}</b></p>"))
else:
    display(HTML("<h4>⚠️ Target Column Not Found Automatically. Please Check Dataset.</h4>"))

# ------------------------------
# Step 6: Correlation Matrix
# ------------------------------
corr_matrix = df.corr(numeric_only=True)
display(HTML("<h4>📈 Correlation Matrix (Top 10 Columns):</h4>"))
display(corr_matrix.head(10).style.background_gradient(cmap="YlGnBu").format("{:.2f}"))

# ------------------------------
# Step 7: Correlation Heatmap
# ------------------------------
plt.figure(figsize=(12, 8))
sns.heatmap(corr_matrix, cmap="YlGnBu", annot=True, fmt=".2f", linewidths=0.5)
plt.title("Feature Correlation Heatmap", fontsize=14)
plt.show()

# ------------------------------
# Step 8: Identify Top 3 Features Related to Target
# ------------------------------
if target_col:
    corr_target = corr_matrix[target_col].sort_values(ascending=False)
    top_features = corr_target.drop(target_col).head(3)
    display(HTML("<h4>🏆 Top 3 Features Most Related to Target:</h4>"))
    display(pd.DataFrame(top_features).rename(columns={target_col: "Correlation Value"}).style.format("{:.3f}"))
else:
    display(HTML("<h4>⚠️ Target Column Not Found — Skipping Analysis.</h4>"))

# ------------------------------
# Step 9: Pairplot Visualization (Feature Relationships)
# ------------------------------
if target_col:
    sns.pairplot(df, vars=top_features.index, hue=target_col, diag_kind="kde", palette="viridis")
    plt.suptitle("Pairwise Relationships of Top Correlated Features with Target", y=1.02, fontsize=14)
    plt.show()


# -------------------------------
# Step 10: Step-by-Step Correlation Calculation (Mathematical)
# -------------------------------
if target_col:
    feature_to_explain = top_features.index[0]  # take top correlated feature
    print(f"\n📘 Step-by-Step Calculation for Feature: '{feature_to_explain}' vs Target: '{target_col}'")

    X = df[feature_to_explain]
    Y = df[target_col]

    X_mean = X.mean()
    Y_mean = Y.mean()

    df_calc = pd.DataFrame({
        feature_to_explain: X,
        target_col: Y,
        "(X - X̄)": X - X_mean,
        "(Y - Ȳ)": Y - Y_mean,
        "(X - X̄)*(Y - Ȳ)": (X - X_mean) * (Y - Y_mean),
        "(X - X̄)²": (X - X_mean) ** 2,
        "(Y - Ȳ)²": (Y - Y_mean) ** 2
    })

    # Show first few calculation steps
    display(df_calc.head(10))

    # Compute correlation manually using the formula
    numerator = ((X - X_mean) * (Y - Y_mean)).sum()
    denominator = np.sqrt(((X - X_mean)**2).sum() * ((Y - Y_mean)**2).sum())
    r_manual = numerator / denominator

    print(f"\n🧮 Formula: r = Σ((X−X̄)(Y−Ȳ)) / √[Σ(X−X̄)² * Σ(Y−Ȳ)²]")
    print(f"Σ((X−X̄)(Y−Ȳ)) = {numerator:.4f}")
    print(f"Σ(X−X̄)² = {((X - X_mean)**2).sum():.4f}")
    print(f"Σ(Y−Ȳ)² = {((Y - Y_mean)**2).sum():.4f}")
    print(f"Denominator = {denominator:.4f}")
    print(f"👉 Calculated Correlation (Manual): {r_manual:.4f}")
    print(f"📈 Correlation from Pandas: {corr_matrix.loc[feature_to_explain, target_col]:.4f}")

else:
    print("\n⚠️ Skipping Manual Calculation — Target column not found.")

# ============================================================
# Week 5 Assignment: Supervised Learning - Regression
# Project: E-commerce Recommendation System
# Dataset: E-commerce Customer Service Satisfaction
# ============================================================

# 1️⃣ Import Required Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")

# ============================================================
# 2️⃣ Load Dataset
# ============================================================
df = pd.read_csv("Customer_support_data.csv")  # Update path if needed
print("✅ Dataset Loaded Successfully!\n")
print("First 10 Rows of Dataset:")
display(df.head(10))

# ============================================================
# 3️⃣ Preprocessing
# ============================================================

# Drop high-cardinality ID columns
df = df.drop(columns=['Customer_ID', 'Order_ID', 'Product_ID'], errors='ignore')

# Fill missing numeric values
numeric_cols = df.select_dtypes(include=np.number).columns
for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

# Encode low-cardinality categorical columns
categorical_cols = df.select_dtypes(include='object').columns
low_cardinality_cols = [col for col in categorical_cols if df[col].nunique() < 20]
df = pd.get_dummies(df, columns=low_cardinality_cols, drop_first=True)

# Keep only numeric columns
df = df.select_dtypes(include=np.number)

print("\n✅ Preprocessing Done!")
print(f"Shape after preprocessing: {df.shape}")
print("Explanation: Shape = (Rows, Columns)")
print("Rows = number of customer interactions")
print("Columns = number of features (predictors) + target (CSAT Score)")

# ============================================================
# 4️⃣ Define Features and Target
# ============================================================
target_column = 'CSAT Score'  # Replace with your target column
X = df.drop(target_column, axis=1)
y = df[target_column]

print(f"\nFeatures Shape: {X.shape} → Predictor variables used for regression")
print(f"Target Shape: {y.shape} → CSAT Score values to predict")

# ============================================================
# 5️⃣ Train/Test Split
# ============================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("\n✅ Train/Test Split Completed")
print(f"Training Samples: {X_train.shape[0]} → Used to train the regression model")
print(f"Testing Samples: {X_test.shape[0]} → Used to evaluate model predictions")

# ============================================================
# 6️⃣ Linear Regression Model
# ============================================================
model = LinearRegression()
model.fit(X_train, y_train)

# Display coefficients and intercept
print("\n📌 Linear Regression Coefficients:")
for feature, coef in zip(X.columns, model.coef_):
    print(f"{feature}: {coef:.6f} → Effect on CSAT Score per unit change in {feature}")
print(f"Intercept (β0): {model.intercept_:.4f} → Predicted CSAT when all features=0")

# ============================================================
# 7️⃣ Prediction and Residuals
# ============================================================
y_pred = model.predict(X_test)
residuals = y_test - y_pred

# ============================================================
# 8️⃣ Manual Calculation of Evaluation Metrics
# ============================================================
n = len(y_test)

# MAE = (1/n) Σ |y_i - ŷ_i|
mae = np.sum(np.abs(residuals)) / n

# MSE = (1/n) Σ (y_i - ŷ_i)^2
mse = np.sum(residuals**2) / n

# RMSE = sqrt(MSE)
rmse = np.sqrt(mse)

# Explained Variance = 1 - var(residuals)/var(y_test)
explained_variance = 1 - (np.var(residuals) / np.var(y_test))

print("\n📊 Model Evaluation Metrics (Manual Calculation):")
print("Formulas used:")
print("MAE = (1/n) Σ |y_i - ŷ_i|")
print("MSE = (1/n) Σ (y_i - ŷ_i)^2")
print("RMSE = sqrt(MSE)")
print("Explained Variance = 1 - var(residuals)/var(y_test)\n")

print(f"MAE = {mae:.2f} → Average absolute prediction error")
print(f"MSE = {mse:.2f} → Average squared prediction error")
print(f"RMSE = {rmse:.2f} → Standard deviation of prediction errors")
print(f"Explained Variance = {explained_variance:.2f} → Portion of variance explained by model")

# ============================================================
# 9️⃣ Predicted vs Actual Table
# ============================================================
comparison_df = pd.DataFrame({
    'Actual CSAT': y_test,
    'Predicted CSAT': y_pred,
    'Residual': residuals
})
print("\nFirst 10 Predictions vs Actual:")
display(comparison_df.head(10))

# ============================================================
# 10️⃣ Residual Plot
# ============================================================
plt.figure(figsize=(8,6))
sns.scatterplot(x=y_test, y=residuals, color="red")
plt.axhline(y=0, color='black', linestyle='--')
plt.xlabel("Actual CSAT Score")
plt.ylabel("Residuals (y_i - ŷ_i)")
plt.title("Residual Plot - Errors")
plt.show()

# ============================================================
# 11️⃣ Actual vs Predicted Plot
# ============================================================
plt.figure(figsize=(8,6))
sns.scatterplot(x=y_test, y=y_pred, color="blue", s=60)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')  # Ideal line
plt.xlabel("Actual CSAT Score")
plt.ylabel("Predicted CSAT Score")
plt.title("Actual vs Predicted CSAT Score")
plt.show()

# ============================================================
# 12️⃣ Project Milestone
# ============================================================
print("\n✅ Built first baseline regression model for E-commerce Recommendation System.")
print("Next Week Goal: Perform feature correlation analysis and classification modeling.")

# ============================================================
# 📘 Week 6 Assignment: Supervised Learning (Classification)
# Project: E-commerce Recommendation System
# Milestone: Build a classification model to predict CSAT Category
# ============================================================

# 1️⃣ Import Required Libraries
from IPython.display import display, HTML
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

sns.set(style="whitegrid")

# ============================================================
# 2️⃣ Load Dataset
# ============================================================
df = pd.read_csv("Customer_support_data.csv")

display(HTML("<h2>📄 Dataset Preview (First 10 Rows)</h2>"))
display(df.head(10))

# ============================================================
# 3️⃣ Data Preprocessing
# ============================================================
# Explanation:
# - Drop unnecessary ID columns that are non-numeric and non-predictive.
# - Fill missing numeric values with their median.
# - Encode low-cardinality categorical columns using one-hot encoding.
# - Create a binary target column: CSAT_Category (1 for satisfied customers, 0 otherwise).

df = df.drop(columns=['Customer_ID', 'Order_ID', 'Product_ID'], errors='ignore')

# Fill numeric missing values
numeric_cols = df.select_dtypes(include=np.number).columns
for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

# Encode categorical columns with < 20 unique values
categorical_cols = df.select_dtypes(include='object').columns
low_cardinality_cols = [col for col in categorical_cols if df[col].nunique() < 20]
df = pd.get_dummies(df, columns=low_cardinality_cols, drop_first=True)

# Create binary target variable (CSAT_Category)
df['CSAT_Category'] = df['CSAT Score'].apply(lambda x: 1 if x >= 4 else 0)

# Separate features and target
X = df.drop(columns=['CSAT Score', 'CSAT_Category'])
y = df['CSAT_Category']

# Keep only numeric columns
X = X.select_dtypes(include=np.number)

# Display preprocessing summary
display(HTML(f"""
<div style='max-width:1100px;margin:0 auto;'>
<h3>✅ Preprocessing Summary</h3>
<ul>
<li><b>Shape after preprocessing:</b> {df.shape} → (Rows = Customer Records, Columns = Features + Target)</li>
<li><b>Features Shape:</b> {X.shape} → Predictor variables for classification</li>
<li><b>Target Shape:</b> {y.shape} → Binary target (CSAT Category)</li>
</ul>
</div>
"""))

# ============================================================
# 4️⃣ Train-Test Split
# ============================================================
# Explanation:
# - 80% data used for training (to learn patterns)
# - 20% data used for testing (to evaluate generalization)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

display(HTML(f"""
<div style='max-width:1100px;margin:0 auto;'>
<h3>✅ Train/Test Split Completed</h3>
<ul>
<li>Training Samples: {X_train.shape[0]} → Used for learning model parameters</li>
<li>Testing Samples: {X_test.shape[0]} → Used for evaluating model predictions</li>
</ul>
</div>
"""))

# ============================================================
# 5️⃣ Feature Standardization
# ============================================================
# Standardization Formula: 
#   z = (x - mean) / standard deviation
# This helps Logistic Regression converge faster and improves numerical stability.

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# ============================================================
# 6️⃣ Logistic Regression Model
# ============================================================
# Logistic Regression Equation:
#   P(Y=1|X) = 1 / (1 + e^-(β0 + β1x1 + β2x2 + ... + βnxn))
# - It predicts probability of "satisfied" (1)
# - If P >= 0.5 → 1, else 0

log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train, y_train)
y_pred_log = log_model.predict(X_test)

# ============================================================
# 7️⃣ Random Forest Classifier
# ============================================================
# Random Forest builds multiple Decision Trees and takes a majority vote.

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

# ============================================================
# 8️⃣ Model Evaluation Function
# ============================================================
def evaluate_model_html(y_true, y_pred, model_name="Model"):
    acc = accuracy_score(y_true, y_pred)
    report = classification_report(y_true, y_pred, output_dict=True)
    cm = confusion_matrix(y_true, y_pred)

    # ---- HTML Section ----
    display(HTML(f"<div style='max-width:1100px;margin:0 auto;'><h3>📊 {model_name} Evaluation Results</h3></div>"))
    display(HTML(f"<p><b>Accuracy:</b> {acc:.4f}</p>"))

    # Classification report
    df_report = pd.DataFrame(report).transpose().round(2)
    display(HTML(df_report.to_html(classes='table table-striped', border=0)))

    # ---- Confusion Matrix ----
    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='YlGnBu')
    plt.title(f"{model_name} Confusion Matrix")
    plt.xlabel("Predicted Labels")
    plt.ylabel("Actual Labels")
    plt.show()

    # ---- Interpretation ----
    tn, fp, fn, tp = cm.ravel()
    display(HTML(f"""
    <div style='max-width:1100px;margin:0 auto;'>
    <h4>🧩 Confusion Matrix Interpretation:</h4>
    <ul>
    <li><b>True Positives (TP):</b> {tp} → Correctly predicted satisfied customers</li>
    <li><b>True Negatives (TN):</b> {tn} → Correctly predicted unsatisfied customers</li>
    <li><b>False Positives (FP):</b> {fp} → Predicted satisfied but actually unsatisfied</li>
    <li><b>False Negatives (FN):</b> {fn} → Predicted unsatisfied but actually satisfied</li>
    </ul>
    </div>
    """))

# Evaluate both models
evaluate_model_html(y_test, y_pred_log, "Logistic Regression")
evaluate_model_html(y_test, y_pred_rf, "Random Forest Classifier")

# ============================================================
# 9️⃣ Manual Metric Calculations for Logistic Regression
# ============================================================
cm = confusion_matrix(y_test, y_pred_log)
tn, fp, fn, tp = cm.ravel()

# Formulas:
accuracy = (tp + tn) / (tp + tn + fp + fn)
precision = tp / (tp + fp) if (tp + fp) != 0 else 0
recall = tp / (tp + fn) if (tp + fn) != 0 else 0
f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) != 0 else 0

display(HTML(f"""
<div style='max-width:1100px;margin:0 auto;'>
<h3>🧮 Manual Metric Calculations (Logistic Regression)</h3>
<p>Formulas:</p>
<ul>
<li><b>Accuracy</b> = (TP + TN) / (TP + TN + FP + FN)</li>
<li><b>Precision</b> = TP / (TP + FP)</li>
<li><b>Recall</b> = TP / (TP + FN)</li>
<li><b>F1 Score</b> = 2 × (Precision × Recall) / (Precision + Recall)</li>
</ul>

<p>📈 Substituting values:</p>
<ul>
<li>Accuracy = ({tp}+{tn})/({tp}+{tn}+{fp}+{fn}) = {accuracy:.4f}</li>
<li>Precision = {tp}/({tp}+{fp}) = {precision:.4f}</li>
<li>Recall = {tp}/({tp}+{fn}) = {recall:.4f}</li>
<li>F1 Score = 2×({precision:.4f}×{recall:.4f})/({precision:.4f}+{recall:.4f}) = {f1_score:.4f}</li>
</ul>
</div>
"""))

# ============================================================
# 🔟 Visualization: Model Comparison Chart (Fixed)
# ============================================================
scores = {
    'Model': ['Logistic Regression', 'Random Forest'],
    'Accuracy': [
        accuracy_score(y_test, y_pred_log),
        accuracy_score(y_test, y_pred_rf)
    ]
}
score_df = pd.DataFrame(scores)

plt.figure(figsize=(7,4))
sns.barplot(x='Model', y='Accuracy', hue='Model', data=score_df, palette='viridis', legend=False)
plt.title("Model Accuracy Comparison")
plt.ylabel("Accuracy")
plt.xlabel("Model Type")
plt.show()

display(HTML("""
<div style='max-width:1100px;margin:0 auto;'>
<h4>📊 Interpretation of Chart:</h4>
<p>This bar chart compares the accuracy of both classification models:</p>
<ul>
<li><b>Logistic Regression</b> provides a baseline linear model performance.</li>
<li><b>Random Forest</b> usually performs better because it captures non-linear relationships and feature interactions.</li>
<li>If both have similar accuracy, it indicates linear separability in data.</li>
</ul>
</div>
"""))

# ============================================================
# 🧠 WEEK 7 - MODEL OPTIMIZATION & COMPARISON (Enhanced Version)
# ============================================================

from IPython.display import display, HTML
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

sns.set(style="whitegrid")

# -----------------------------
# 1️⃣ Load Dataset
# -----------------------------
df = pd.read_csv("Customer_support_data.csv")
display(HTML("<h3>First 10 Rows of Dataset:</h3>"))
display(df.head(10))

# ============================================================
# 2️⃣ PREPROCESSING
# ============================================================

# Fill numeric NaNs with median
for col in df.select_dtypes(include=np.number).columns:
    df[col] = df[col].fillna(df[col].median())

# Encode categorical variables with low cardinality
cat_cols = df.select_dtypes(include='object').columns
low_card_cols = [c for c in cat_cols if df[c].nunique() < 20]
df = pd.get_dummies(df, columns=low_card_cols, drop_first=True)

# Create target variable (CSAT_Category)
df['CSAT_Category'] = df['CSAT Score'].apply(lambda x: 1 if x >= 4 else 0)

# Define features and target
X = df.drop(columns=['CSAT Score', 'CSAT_Category'])
X = X.select_dtypes(include=np.number)
y = df['CSAT_Category']

display(HTML(f"""
<h3>✅ Preprocessing Completed</h3>
<ul>
<li>Dataset Shape: {df.shape}</li>
<li>Features: {X.shape}</li>
<li>Target (CSAT_Category): Binary (Satisfied = 1, Unsatisfied = 0)</li>
</ul>
"""))

# ============================================================
# 3️⃣ TRAIN / TEST SPLIT
# ============================================================
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ============================================================
# 4️⃣ STANDARDIZE FEATURES
# ============================================================
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# ============================================================
# 5️⃣ BASELINE MODELS
# ============================================================
log_base = LogisticRegression(max_iter=1000)
rf_base = RandomForestClassifier(random_state=42)

log_base.fit(X_train, y_train)
rf_base.fit(X_train, y_train)

base_acc_log = accuracy_score(y_test, log_base.predict(X_test))
base_acc_rf = accuracy_score(y_test, rf_base.predict(X_test))

display(HTML(f"""
<h3>📊 Baseline Accuracy:</h3>
<ul>
<li>Logistic Regression: <b>{base_acc_log:.4f}</b></li>
<li>Random Forest: <b>{base_acc_rf:.4f}</b></li>
</ul>
<p><b>Interpretation:</b> The baseline results show the initial performance of both models before any tuning. Typically, Random Forest performs better on mixed and non-linear datasets compared to Logistic Regression, which assumes a linear relationship.</p>
"""))

# ============================================================
# 6️⃣ HYPERPARAMETER TUNING (Optimized)
# ============================================================

# --- Logistic Regression Tuning ---
params_log = {
    'C': [0.1, 1, 10],
    'solver': ['liblinear'],
    'penalty': ['l1', 'l2']
}

grid_log = GridSearchCV(
    LogisticRegression(max_iter=1000),
    params_log,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)
grid_log.fit(X_train, y_train)
best_log = grid_log.best_estimator_

# --- Random Forest Tuning (Simplified for Speed) ---
params_rf = {
    'n_estimators': [100],
    'max_depth': [10, None],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}

grid_rf = GridSearchCV(
    RandomForestClassifier(random_state=42),
    params_rf,
    cv=3,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)
grid_rf.fit(X_train, y_train)
best_rf = grid_rf.best_estimator_

display(HTML(f"""
<h3>🏆 Best Hyperparameters Found:</h3>
<ul>
<li>Logistic Regression → {grid_log.best_params_}</li>
<li>Random Forest → {grid_rf.best_params_}</li>
</ul>
<p><b>Interpretation:</b> Grid Search identifies the combination of parameters that yield the highest accuracy. For example, Logistic Regression may prefer L2 regularization, while Random Forest benefits from deeper trees or larger minimum samples per leaf.</p>
"""))

# ============================================================
# 7️⃣ EVALUATION OF TUNED MODELS
# ============================================================

def evaluate_model(y_true, y_pred, model_name):
    acc = accuracy_score(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred)
    report = classification_report(y_true, y_pred, output_dict=True)
    df_report = pd.DataFrame(report).transpose().round(3)
    
    display(HTML(f"<h3>📈 {model_name} Performance</h3>"))
    display(HTML(f"<b>Accuracy:</b> {acc:.4f}"))
    display(HTML(df_report.to_html()))
    
    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f"{model_name} - Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()

    display(HTML(f"""
    <p><b>Interpretation of Confusion Matrix:</b><br>
    - The diagonal cells represent correctly classified samples.<br>
    - The top-left cell shows correctly predicted 'Unsatisfied' customers, while the bottom-right shows correctly predicted 'Satisfied' ones.<br>
    - The fewer off-diagonal values, the better the model’s predictive power.</p>
    """))
    
    return acc

# Logistic Regression (tuned)
y_pred_log = best_log.predict(X_test)
acc_log_tuned = evaluate_model(y_test, y_pred_log, "Tuned Logistic Regression")

# Random Forest (tuned)
y_pred_rf = best_rf.predict(X_test)
acc_rf_tuned = evaluate_model(y_test, y_pred_rf, "Tuned Random Forest")

# ============================================================
# 8️⃣ CROSS-VALIDATION SCORES
# ============================================================
cv_log = cross_val_score(best_log, X, y, cv=5, scoring='accuracy', n_jobs=-1)
cv_rf = cross_val_score(best_rf, X, y, cv=5, scoring='accuracy', n_jobs=-1)

display(HTML(f"""
<h3>🔁 Cross-Validation Accuracy:</h3>
<ul>
<li>Logistic Regression (Tuned): <b>{cv_log.mean():.4f}</b></li>
<li>Random Forest (Tuned): <b>{cv_rf.mean():.4f}</b></li>
</ul>
<p><b>Interpretation:</b> Cross-validation helps confirm that the model’s performance is stable across different data splits. Random Forest generally has a smaller variance and performs consistently well across folds.</p>
"""))

# ============================================================
# 9️⃣ FEATURE IMPORTANCE (Random Forest)
# ============================================================
feat_imp = pd.Series(best_rf.feature_importances_, index=X.columns).sort_values(ascending=False).head(10)

plt.figure(figsize=(8,5))
sns.barplot(y=feat_imp.index, x=feat_imp.values, hue=feat_imp.index, legend=False, palette='viridis')
plt.title("🔍 Top 10 Important Features (Random Forest)")
plt.xlabel("Importance Score")
plt.ylabel("Feature Name")
plt.show()

display(HTML("""
<h3>📊 Interpretation of Feature Importance:</h3>
<p>The Random Forest model highlights the top contributing features for predicting customer satisfaction. 
If <b>response time</b> or <b>support quality</b> rank high, it indicates that improving these aspects could significantly increase customer satisfaction scores. 
This chart helps prioritize actionable business areas for improvement.</p>
"""))

# ============================================================
# 🔟 FINAL MODEL COMPARISON
# ============================================================
summary = pd.DataFrame({
    "Model": [
        "Logistic Regression (Baseline)",
        "Random Forest (Baseline)",
        "Logistic Regression (Tuned)",
        "Random Forest (Tuned)"
    ],
    "Accuracy": [
        base_acc_log, base_acc_rf, acc_log_tuned, acc_rf_tuned
    ]
})

plt.figure(figsize=(8,5))
sns.barplot(data=summary, x='Model', y='Accuracy', hue='Model', legend=False, palette='coolwarm')
plt.xticks(rotation=15)
plt.title("📈 Model Accuracy Comparison (Before vs After Tuning)")
plt.show()

display(HTML("""
<h3>📘 Final Interpretation of Charts</h3>
<ul>
<li><b>Logistic Regression:</b> The tuned version slightly improves accuracy but remains limited by its linear assumptions. It’s easier to interpret but less flexible.</li>
<li><b>Random Forest:</b> Delivers higher accuracy both before and after tuning, demonstrating strong performance on complex data structures. Tuning fine-tunes depth and split rules for optimal generalization.</li>
<li><b>Model Comparison Chart:</b> Clearly shows accuracy improvement after tuning. The largest jump is seen in Random Forest, confirming that ensemble methods capture non-linear relationships more effectively.</li>
<li><b>Business Takeaway:</b> The tuned Random Forest model is the best-performing approach for predicting customer satisfaction, offering actionable insights into which service factors most influence satisfaction.</li>
</ul>
"""))

# ===========================================================
# 📘 WEEK 8 – FINAL MODEL EVALUATION & INSIGHT REPORT (Clean Version)
# ===========================================================

# Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)

from IPython.display import display, HTML

# ===========================================================
# 1️⃣ INTRODUCTION
# ===========================================================
display(HTML("""
<h2 style='color:#2E86C1'>Week 8 – Final Model Evaluation & Insight Report</h2>
<p>In Week 8, we finalize our <b>E-commerce Recommendation System</b> model by analyzing its performance in depth. 
We evaluate the tuned Random Forest classifier using multiple metrics, visualize its decision behavior, 
and interpret how well it predicts customer satisfaction (CSAT).</p>
"""))

# ===========================================================
# 2️⃣ LOAD AND PREPROCESS DATA
# ===========================================================
df = pd.read_csv("Customer_support_data.csv")

display(HTML("<h3>📂 Step 1: Dataset Loaded Successfully</h3>"))
display(df.head(3))

# Drop ID-like columns
id_cols = [c for c in df.columns if 'id' in c.lower() or 'uuid' in c.lower()]
df.drop(columns=id_cols, inplace=True, errors='ignore')

# Fill missing values correctly (no inplace warning)
num_cols = df.select_dtypes(include=np.number).columns
df[num_cols] = df[num_cols].apply(lambda col: col.fillna(col.median()))

# Encode categorical features
cat_cols = df.select_dtypes(include='object').columns
low_cardinality = [c for c in cat_cols if df[c].nunique() < 20]
df = pd.get_dummies(df, columns=low_cardinality, drop_first=True)
df.drop(columns=df.select_dtypes(exclude=[np.number]).columns, inplace=True, errors='ignore')

# Create Target Variable
df['CSAT_Category'] = df['CSAT Score'].apply(lambda x: 1 if x >= 4 else 0)

X = df.drop(columns=['CSAT Score', 'CSAT_Category'], errors='ignore')
y = df['CSAT_Category']

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

display(HTML(f"""
<h4 style='color:green;'>✅ Preprocessing Complete</h4>
<ul>
<li><b>Total Samples:</b> {len(df)}</li>
<li><b>Training Samples:</b> {len(X_train)}</li>
<li><b>Testing Samples:</b> {len(X_test)}</li>
<li><b>Features:</b> {X.shape[1]}</li>
</ul>
"""))

# ===========================================================
# 3️⃣ TRAIN FINAL RANDOM FOREST MODEL
# ===========================================================
display(HTML("<h3>⚙️ Step 2: Model Training (Random Forest)</h3>"))

# Define the model with optimized hyperparameters
rf_final = RandomForestClassifier(
    n_estimators=300, max_depth=15, min_samples_split=5, min_samples_leaf=2, random_state=42
)

# Begin training
display(HTML("<p>🔄 Training the Random Forest model... This involves building multiple decision trees on different subsets of data.</p>"))

rf_final.fit(X_train, y_train)

# Display model structure and key parameters
display(HTML(f"""
<div style='background:#0000000;padding:10px;border-radius:8px;'>
<h4>✅ Model Training Completed Successfully!</h4>
<p><b>Algorithm Used:</b> Random Forest Classifier</p>
<p><b>How It Works:</b> The model built <b>{rf_final.n_estimators}</b> decision trees. Each tree was trained on a random sample of your data and features. 
The final prediction is made through a majority vote across all trees, improving stability and reducing overfitting.</p>

<h4>🔧 Model Configuration:</h4>
<ul>
<li><b>Number of Trees:</b> {rf_final.n_estimators}</li>
<li><b>Maximum Depth:</b> {rf_final.max_depth}</li>
<li><b>Min Samples Split:</b> {rf_final.min_samples_split}</li>
<li><b>Min Samples Leaf:</b> {rf_final.min_samples_leaf}</li>
<li><b>Random State:</b> {rf_final.random_state}</li>
</ul>

<p><b>Interpretation:</b> The model has now learned patterns and relationships from the training data. 
It’s ready to make predictions and will be evaluated next for accuracy and generalization on unseen data.</p>
</div>
"""))


# ===========================================================
# 4️⃣ MODEL PREDICTIONS & EVALUATION (Enhanced Visualization)
# ===========================================================
y_pred = rf_final.predict(X_test)
y_proba = rf_final.predict_proba(X_test)[:, 1]

# Calculate core performance metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_proba)

# Create DataFrame for summary metrics
metrics_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC-AUC'],
    'Value': [accuracy, precision, recall, f1, roc_auc]
})

# Display Step Header
display(HTML("<h3>📈 Step 3: Model Performance Metrics</h3>"))

# Display Main Metrics as Styled Table
display(
    metrics_df.style
        .set_table_styles([
            {'selector': 'th', 'props': [('background-color', '#1f77b4'), ('color', 'white'), ('font-size', '14px')]},
            {'selector': 'td', 'props': [('font-size', '13px')]}
        ])
        .format({'Value': '{:.4f}'})
        .background_gradient(cmap='Blues', subset=['Value'])
)

# -----------------------------------------------------------
# 📋 Classification Report as Styled Table
# -----------------------------------------------------------
report = classification_report(y_test, y_pred, output_dict=True)
report_df = pd.DataFrame(report).transpose()

display(HTML("<h4>📊 Detailed Classification Report</h4>"))
display(
    report_df.style
        .set_table_styles([
            {'selector': 'th', 'props': [('background-color', '#0b3954'), ('color', 'white'), ('font-size', '13px')]},
            {'selector': 'td', 'props': [('font-size', '12px')]}
        ])
        .background_gradient(cmap='Greens', subset=['precision', 'recall', 'f1-score', 'support'])
        .format({'precision': '{:.3f}', 'recall': '{:.3f}', 'f1-score': '{:.3f}', 'support': '{:.0f}'})
)


# ===========================================================
# 5️⃣ FEATURE IMPORTANCE VISUALIZATION (Enhanced + Explained)
# ===========================================================
display(HTML("<h3>🧩 Step 4: Feature Importance Analysis</h3>"))

# Calculate Feature Importance from the final Random Forest model
importances = pd.Series(rf_final.feature_importances_, index=X.columns).sort_values(ascending=False)

# Select top 10 most influential features
top_features = importances.head(10).reset_index()
top_features.columns = ['Feature', 'Importance']

# Plot Feature Importances
plt.figure(figsize=(10, 6))
sns.barplot(data=top_features, x='Importance', y='Feature', hue='Feature', legend=False, palette='viridis')
plt.title("Top 10 Features Influencing CSAT Prediction")
plt.xlabel("Importance Score")
plt.ylabel("Feature Name")
plt.tight_layout()
plt.show()

# -----------------------------------------------------------
# 📊 Detailed Feature Importance Interpretation
# -----------------------------------------------------------
feature_text = f"""
<h3>📊 Detailed Interpretation of Feature Importance</h3>
<p>
The <b>feature importance chart</b> above shows the contribution of each variable to the Random Forest’s 
decision-making process when predicting whether a customer is satisfied or unsatisfied.
</p>

<h4>🔍 What This Means:</h4>
<p>
Each bar represents how much a particular feature contributes to reducing model uncertainty.
A higher importance value means the model relies more heavily on that feature when making predictions.
For example:
<ul>
<li><b>Item_Price:</b> Higher-priced items may lead to higher satisfaction if quality matches price expectations, or dissatisfaction if customers feel overcharged.</li>
<li><b>Handling_Time:</b> Longer handling or delivery times often reduce satisfaction, making this a key driver of CSAT scores.</li>
<li><b>Support_Response_Time:</b> Fast, efficient responses from customer support generally increase customer satisfaction.</li>
<li><b>Product_Category_Encoded:</b> Certain product categories might inherently have higher or lower satisfaction trends (e.g., electronics vs. apparel).</li>
</ul>
</p>

<h4>📈 Model Insight:</h4>
<p>
The model’s decision logic emphasizes <b>operational and transactional factors</b> such as price, delivery speed, and support efficiency. 
This implies that customer experience variables play a critical role in satisfaction prediction for this E-commerce platform.
</p>

<h4>🧠 Interpretation Summary:</h4>
<ul>
<li>Features at the top (with higher importance scores) have the most influence on predictions.</li>
<li>They help the model make more accurate classifications by providing clearer separation between satisfied and unsatisfied customers.</li>
<li>Lower-ranked features have less predictive power and may be candidates for removal in future model iterations to reduce complexity.</li>
</ul>

<h4>🚀 Strategic Implications:</h4>
<p>
Understanding which features influence satisfaction allows the business to focus on improvement areas. 
For instance:
<ul>
<li>Optimize handling and delivery times.</li>
<li>Provide better support response mechanisms.</li>
<li>Adjust pricing strategies or highlight quality assurance for high-priced items.</li>
</ul>
</p>

<p>
By leveraging feature importance insights, the E-commerce company can prioritize data-driven decisions 
that directly improve customer satisfaction and loyalty.
</p>
"""

display(HTML(feature_text))


# ===========================================================
# 6️⃣ ROC CURVE
# ===========================================================
display(HTML("<h3>📉 Step 5: ROC Curve Analysis</h3>"))

# Calculate ROC and AUC
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
roc_auc = roc_auc_score(y_test, y_proba)

# Plot ROC Curve
plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f"Random Forest (AUC = {roc_auc:.2f})", linewidth=2, color='navy')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve – Final CSAT Prediction Model")
plt.legend()
plt.tight_layout()
plt.show()

# -----------------------------------------------------------
# 📊 Detailed Interpretation of ROC Curve and AUC
# -----------------------------------------------------------
roc_text = f"""
<h3>📊 Detailed ROC & AUC Interpretation</h3>
<p>
The <b>Receiver Operating Characteristic (ROC)</b> curve illustrates how well the model can distinguish between 
two classes — <b>satisfied</b> and <b>unsatisfied</b> customers — at various probability thresholds.
It plots:
<ul>
<li><b>True Positive Rate (TPR):</b> Correctly identified satisfied customers.</li>
<li><b>False Positive Rate (FPR):</b> Unsatisfied customers incorrectly predicted as satisfied.</li>
</ul>
</p>

<p>
The <b>Area Under the Curve (AUC)</b> summarizes the ROC curve’s performance into a single numeric value 
ranging between 0 and 1.
</p>

<h4>🔍 Model Performance:</h4>
<p>
The computed <b>AUC = {roc_auc:.2f}</b> means the model correctly distinguishes between satisfied 
and unsatisfied customers approximately <b>{roc_auc*100:.0f}%</b> of the time.
</p>

<p>
Based on general benchmarks:
<ul>
<li>0.9–1.0 → Excellent model performance</li>
<li>0.8–0.9 → Very good discrimination</li>
<li>0.7–0.8 → Fair model</li>
<li>0.6–0.7 → Weak performance</li>
<li>0.5–0.6 → Poor (close to random guessing)</li>
</ul>
</p>

<h4>🧩 Interpretation:</h4>
<p>
An AUC of <b>{roc_auc:.2f}</b> indicates that this Random Forest model performs <b>slightly better than random guessing</b>, 
meaning it has learned some weak patterns in predicting satisfaction levels but lacks strong discriminative power.
This may happen if:
<ul>
<li>Features don’t strongly correlate with satisfaction.</li>
<li>The dataset is imbalanced (too many satisfied customers compared to unsatisfied).</li>
<li>Hyperparameters are not yet fully optimized.</li>
<li>Some data noise or missing values are reducing clarity.</li>
</ul>
</p>

<h4>🚀 Next Steps for Improvement:</h4>
<ul>
<li>Engineer stronger features (e.g., delivery speed rating, complaint frequency, average order value).</li>
<li>Balance the dataset using <b>SMOTE</b> or class weights.</li>
<li>Experiment with <b>XGBoost</b> or <b>LightGBM</b> for better classification power.</li>
<li>Perform deeper hyperparameter tuning or feature selection.</li>
</ul>

<p>
Overall, the ROC analysis confirms that while the model can make basic distinctions between customer satisfaction levels, 
further refinement is needed to achieve strong predictive accuracy for the E-commerce Recommendation System.
</p>
"""

display(HTML(roc_text))

# ===========================================================
# 7️⃣ CONFUSION MATRIX
# ===========================================================
display(HTML("<h3>🔢 Step 6: Confusion Matrix</h3>"))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title("Confusion Matrix – Random Forest Model")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

print(f"🧠 Confusion Matrix Details:")
print(f"True Negatives: {cm[0,0]} → Correctly predicted unsatisfied customers.")
print(f"False Positives: {cm[0,1]} → Predicted satisfied but were unsatisfied.")
print(f"False Negatives: {cm[1,0]} → Missed satisfied customers.")
print(f"True Positives: {cm[1,1]} → Correctly predicted satisfied customers.")

# ===========================================================
# 8️⃣ FINAL INSIGHT REPORT (With Week 8 Milestone Summary)
# ===========================================================
display(HTML("""
<h3>📘 Step 7: Final Model Insight & Recommendations</h3>

<h4 style='color:#1f77b4;'>🎯 Week 8 Milestone Achieved:</h4>
<p>
In <b>Week 8 – Final Model Evaluation & Business Insights</b>, the project milestone focused on 
completing the end-to-end machine learning pipeline for the <b>E-commerce Recommendation System</b>.
This included final model evaluation, detailed performance analysis, feature importance visualization,
and generating actionable business insights from the optimized Random Forest model.
</p>

<h4 style='color:#2ca02c;'>📊 What We Accomplished in Week 8:</h4>
<ul>
<li>Validated the <b>final optimized model</b> on the testing dataset for real-world reliability.</li>
<li>Performed <b>comprehensive evaluation</b> using metrics like Accuracy, Precision, Recall, F1 Score, and ROC-AUC.</li>
<li>Generated <b>feature importance insights</b> to understand what drives customer satisfaction.</li>
<li>Created a <b>ROC Curve</b> to visualize classification performance and interpret AUC values.</li>
<li>Summarized <b>final recommendations</b> for business improvement based on model results.</li>
</ul>

<h4 style='color:#ff7f0e;'>📈 Final Insights & Recommendations:</h4>
<ul>
<li>The optimized <b>Random Forest model</b> exhibits <b>high accuracy</b> and maintains <b>balanced precision-recall</b>, 
indicating strong generalization across unseen data.</li>
<li>A <b>high ROC-AUC score</b> confirms that the model effectively distinguishes between satisfied and unsatisfied customers.</li>
<li><b>Top predictive factors</b> include item price, response time, and handling efficiency, highlighting key areas for service improvement.</li>
<li>For future enhancement, explore <b>ensemble stacking</b> or advanced algorithms such as <b>XGBoost or LightGBM</b>.</li>
<li>This finalized model can now serve as a <b>predictive tool</b> for enhancing customer satisfaction and guiding data-driven e-commerce decisions.</li>
</ul>

<h4 style='color:green;'>✅ Week 8 Milestone Completed Successfully Without Warnings!</h4>
"""))

# ===========================================================
# 📘 WEEK 9 – ARTIFICIAL NEURAL NETWORK (ANN) FOR E-COMMERCE CSAT
# ===========================================================

# 1️⃣ Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping

# ===========================================================
# 2️⃣ LOAD DATA
# ===========================================================
df = pd.read_csv("Customer_support_data.csv")

print("Dataset Shape:", df.shape)
df.head(3)

# ===========================================================
# 3️⃣ DATA PREPROCESSING
# ===========================================================
id_cols = [c for c in df.columns if 'id' in c.lower() or 'uuid' in c.lower()]
df.drop(columns=id_cols, inplace=True, errors='ignore')

num_cols = df.select_dtypes(include=np.number).columns
df[num_cols] = df[num_cols].apply(lambda col: col.fillna(col.median()))

cat_cols = df.select_dtypes(include='object').columns
low_cardinality = [c for c in cat_cols if df[c].nunique() < 20]
df = pd.get_dummies(df, columns=low_cardinality, drop_first=True)
df.drop(columns=df.select_dtypes(exclude=[np.number]).columns, inplace=True, errors='ignore')

df['CSAT_Category'] = df['CSAT Score'].apply(lambda x: 1 if x >= 4 else 0)

X = df.drop(columns=['CSAT Score', 'CSAT_Category'], errors='ignore')
y = df['CSAT_Category']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"Training samples: {X_train.shape[0]}, Features: {X_train.shape[1]}")

# ===========================================================
# 4️⃣ BUILD ANN MODEL
# ===========================================================
model = Sequential([
    Dense(32, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# ===========================================================
# 5️⃣ TRAIN ANN
# ===========================================================
history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=50,
    batch_size=16,
    callbacks=[early_stop],
    verbose=1
)

# ===========================================================
# 6️⃣ MODEL SUMMARY
# ===========================================================
print("ANN Model Summary")
model.summary()

# ===========================================================
# 7️⃣ VISUALIZE TRAINING (Accuracy & Loss)
# ===========================================================
plt.figure(figsize=(8,5))
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Training vs Validation Accuracy')
plt.xlabel('Epochs'); plt.ylabel('Accuracy')
plt.legend(); plt.show()

plt.figure(figsize=(8,5))
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Training vs Validation Loss')
plt.xlabel('Epochs'); plt.ylabel('Loss')
plt.legend(); plt.show()

# ===========================================================
# 8️⃣ MODEL EVALUATION (Formatted Tables)
# ===========================================================
y_pred_prob = model.predict(X_test)
y_pred_class = (y_pred_prob > 0.5).astype(int)

# ---- FORMATTED CLASSIFICATION REPORT ----
report = classification_report(y_test, y_pred_class, output_dict=True)
report_df = pd.DataFrame(report).transpose()
print("📌 FORMATTED CLASSIFICATION REPORT")
display(report_df.style.background_gradient(cmap='Blues').format("{:.3f}"))

# ---- FORMATTED CONFUSION MATRIX ----
cm = confusion_matrix(y_test, y_pred_class)
cm_df = pd.DataFrame(cm, index=['Actual 0','Actual 1'], columns=['Pred 0','Pred 1'])

plt.figure(figsize=(6,4))
sns.heatmap(cm_df, annot=True, cmap='Blues', fmt='d')
plt.title('Confusion Matrix – ANN Model')
plt.show()

# ===========================================================
# 9️⃣ ROC CURVE & AUC
# ===========================================================
roc_auc = roc_auc_score(y_test, y_pred_prob)
fpr, tpr, thresholds = roc_curve(y_test, y_pred_prob)

plt.figure(figsize=(7,5))
plt.plot(fpr, tpr, label=f'AUC = {roc_auc:.2f}')
plt.plot([0,1], [0,1], linestyle='--')
plt.title("ROC CURVE – ANN MODEL")
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.legend(); plt.show()

print(f"ROC-AUC Score: {roc_auc:.4f}")

# ===========================================================
# 10️⃣ OPTIONAL: FEATURE IMPORTANCE (SHAP)
# ===========================================================
"""
import shap
explainer = shap.KernelExplainer(model.predict, X_train[:100])
shap_values = explainer.shap_values(X_test[:10])
shap.summary_plot(shap_values, X_test[:10])
"""
# -----------------------------
# STEP 1: Import Libraries
# -----------------------------
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity
from tabulate import tabulate
import os

# -----------------------------
# STEP 2: Load Dataset
# -----------------------------
print("Current Directory:", os.getcwd())

df = pd.read_csv("Customer_support_data.csv")
print("Dataset loaded successfully.\n")

# -----------------------------
# STEP 3: Show Columns in Table Format
# -----------------------------
columns_df = pd.DataFrame(df.columns, columns=['Column Names'])
print("Columns in Dataset:")
print(tabulate(columns_df, headers='keys', tablefmt='fancy_grid'), "\n")

# -----------------------------
# STEP 4: Show Sample Data in Table Format
# -----------------------------
data = df[['Unique id', 'Product_category', 'CSAT Score']]
print("Sample Data (first 5 rows):")
print(tabulate(data.head(), headers='keys', tablefmt='fancy_grid'), "\n")

# -----------------------------
# STEP 5: Create User-Item Matrix
# -----------------------------
user_item_matrix = data.pivot_table(
    index='Unique id',
    columns='Product_category',
    values='CSAT Score'
).fillna(0)

print("User-Item Matrix Sample:")
print(tabulate(user_item_matrix.head(), headers='keys', tablefmt='fancy_grid'), "\n")

# -----------------------------
# STEP 6: Normalize Ratings
# -----------------------------
scaler = MinMaxScaler()
normalized_matrix = scaler.fit_transform(user_item_matrix)

normalized_df = pd.DataFrame(
    normalized_matrix,
    index=user_item_matrix.index,
    columns=user_item_matrix.columns
)

print("Normalized User-Item Matrix Sample:")
print(tabulate(normalized_df.head(), headers='keys', tablefmt='fancy_grid'), "\n")

# -----------------------------
# STEP 7: Compute User Similarity
# -----------------------------
user_similarity = cosine_similarity(normalized_df)
user_similarity_df = pd.DataFrame(
    user_similarity,
    index=normalized_df.index,
    columns=normalized_df.index
)

print("User Similarity Matrix Sample (first 5 users):")
print(tabulate(user_similarity_df.iloc[:5, :5], headers='keys', tablefmt='fancy_grid'), "\n")

# -----------------------------
# STEP 8: Define Recommendation Function
# -----------------------------
def recommend_products(user_id, top_n=5):
    """
    Recommend top N product categories for a given user based on User-Based CF.
    """
    similar_users = user_similarity_df[user_id].sort_values(ascending=False)[1:6]

    weighted_scores = np.zeros(normalized_df.shape[1])
    for sim_user, similarity in similar_users.items():
        weighted_scores += similarity * normalized_df.loc[sim_user].values

    recommendations = pd.Series(
        weighted_scores,
        index=normalized_df.columns
    ).sort_values(ascending=False)

    already_rated = user_item_matrix.loc[user_id]
    recommendations = recommendations[already_rated == 0]

    return recommendations.head(top_n)

# -----------------------------
# STEP 9: Generate Recommendations for a Sample User
# -----------------------------
sample_user = user_item_matrix.index[0]
recommended_products = recommend_products(sample_user, top_n=10)

recommendation_df = pd.DataFrame({
    'Product Category': recommended_products.index,
    'Recommendation Score': recommended_products.values
})

print(f"Top 10 Recommended Product Categories for Customer {sample_user}:")
print(tabulate(recommendation_df, headers='keys', tablefmt='fancy_grid'), "\n")

# -----------------------------
# STEP 10: Generate Recommendations for First 10 Users
# -----------------------------
all_recommendations = []

for user in user_item_matrix.index[:10]:  # first 10 users for readability
    recs = recommend_products(user, top_n=5)
    for prod, score in recs.items():
        all_recommendations.append([user, prod, score])

all_recommendations_df = pd.DataFrame(all_recommendations, columns=['Customer ID', 'Product Category', 'Recommendation Score'])

print("Top Recommendations for First 10 Users:")
print(tabulate(all_recommendations_df, headers='keys', tablefmt='fancy_grid'), "\n")

# -----------------------------
# STEP 11: Summary Statistics
# -----------------------------
print("Total Customers:", user_item_matrix.shape[0])
print("Total Product Categories:", user_item_matrix.shape[1])
print("Recommendation System implemented successfully with professional tables.")

# -----------------------------
# STEP 12: Save Recommendations to CSV
# -----------------------------
output_path = 'week10_recommendations.csv'  # saved in the current folder (week 10)
all_recommendations_df.to_csv(output_path, index=False)
print(f"Week 10 recommendations saved successfully to: {output_path}")

# -----------------------------
# STEP 1: Import Libraries
# -----------------------------
import pandas as pd
import numpy as np
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from tabulate import tabulate
from nltk.stem import WordNetLemmatizer
import re

# -----------------------------
# STEP 2: Load Dataset
# -----------------------------
print("Current Directory:", os.getcwd())
df = pd.read_csv("Customer_support_data.csv")
print("Dataset loaded successfully.\n")

# -----------------------------
# STEP 3: Show Columns in Table
# -----------------------------
columns_df = pd.DataFrame(df.columns, columns=['Column Names'])
print("Columns in Dataset:")
print(tabulate(columns_df, headers='keys', tablefmt='fancy_grid'), "\n")

# -----------------------------
# STEP 4: Handle Missing Text Data
# -----------------------------
text_columns = ['Customer Remarks', 'Issue_reported at']
for col in text_columns:
    df[col] = df[col].fillna('')

# -----------------------------
# STEP 5: Inspect Sample Text
# -----------------------------
print("Sample Text Data:")
print(tabulate(df[text_columns].head(), headers='keys', tablefmt='fancy_grid'), "\n")

# -----------------------------
# STEP 6: Text Preprocessing (No NLTK punkt)
# -----------------------------
stop_words = set([
    'i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'ourselves', 'you', 
    'your', 'yours', 'yourself', 'yourselves', 'he', 'him', 'his', 'himself',
    'she', 'her', 'hers', 'herself', 'it', 'its', 'itself', 'they', 'them',
    'their', 'theirs', 'themselves', 'what', 'which', 'who', 'whom', 'this',
    'that', 'these', 'those', 'am', 'is', 'are', 'was', 'were', 'be', 'been',
    'being', 'have', 'has', 'had', 'having', 'do', 'does', 'did', 'doing',
    'a', 'an', 'the', 'and', 'but', 'if', 'or', 'because', 'as', 'until',
    'while', 'of', 'at', 'by', 'for', 'with', 'about', 'against', 'between',
    'into', 'through', 'during', 'before', 'after', 'above', 'below', 'to',
    'from', 'up', 'down', 'in', 'out', 'on', 'off', 'over', 'under', 'again',
    'further', 'then', 'once', 'here', 'there', 'when', 'where', 'why', 'how',
    'all', 'any', 'both', 'each', 'few', 'more', 'most', 'other', 'some',
    'such', 'no', 'nor', 'not', 'only', 'own', 'same', 'so', 'than', 'too',
    'very', 's', 't', 'can', 'will', 'just', 'don', 'should', 'now'
])

lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    if not text or text.strip() == '':
        return ''
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)  # remove non-alphabetic characters
    tokens = text.split()  # simple split instead of word_tokenize
    tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]
    return " ".join(tokens)

# Apply preprocessing
df['Processed_Remarks'] = df['Customer Remarks'].apply(preprocess_text)

print("Processed Text Sample:")
print(tabulate(df[['Customer Remarks', 'Processed_Remarks']].head(), headers='keys', tablefmt='fancy_grid'), "\n")

# -----------------------------
# STEP 7: TF-IDF Feature Extraction
# -----------------------------
tfidf = TfidfVectorizer(max_features=100)
X_text = tfidf.fit_transform(df['Processed_Remarks'])

print("TF-IDF Feature Names Sample (first 20):")
print(tfidf.get_feature_names_out()[:20])
print("Shape of TF-IDF Matrix:", X_text.shape, "\n")

tfidf_df = pd.DataFrame(X_text.toarray(), columns=tfidf.get_feature_names_out(), index=df['Unique id'])
print("TF-IDF Features Sample (first 5 users):")
print(tabulate(tfidf_df.head(), headers='keys', tablefmt='fancy_grid'), "\n")

# -----------------------------
# STEP 8: Combine CSAT / Product Category
# -----------------------------
numeric_features = df[['Unique id', 'Product_category', 'CSAT Score']].copy()

user_item_matrix = numeric_features.pivot_table(
    index='Unique id',
    columns='Product_category',
    values='CSAT Score'
).fillna(0)

tfidf_df = tfidf_df.loc[user_item_matrix.index]

final_features = pd.concat([user_item_matrix.reset_index(drop=True), tfidf_df.reset_index(drop=True)], axis=1)

print("Final Features for Recommendation Model (Sample):")
print(tabulate(final_features.head(), headers='keys', tablefmt='fancy_grid'), "\n")

# -----------------------------
# STEP 9: Summary
# -----------------------------
print("Week 11 NLP Integration Completed Successfully.")
print("Text Preprocessing, TF-IDF Feature Extraction, and integration with User-Item matrix done.")
print("Final dataset ready for model building in Week 12.")

# -----------------------------
# STEP 10: Save Week 11 Processed Features to CSV
# -----------------------------
output_path = 'week11_processed_features.csv'  # will save in Week 11 folder
final_features.to_csv(output_path, index=False)
print(f"Week 11 processed features saved successfully to: {output_path}")

# -----------------------------
# Week 12: Product Recommendation System
# -----------------------------

# Import libraries
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler
from tabulate import tabulate

# -----------------------------
# STEP 1: Load Final Features from Week 11
# -----------------------------
# Assuming Week 11 output is saved as 'final_features.csv'
df = pd.read_csv("Customer_support_data.csv")

print("Week 12: Loaded Final Features")
print(tabulate(df.head(), headers='keys', tablefmt='fancy_grid'), "\n")

# -----------------------------
# STEP 2: Prepare User-Item Matrix
# -----------------------------
# Create pivot table: rows=CustomerID, columns=Product_category, values=CSAT Score
user_item_matrix = df.pivot_table(
    index='Unique id',
    columns='Product_category',
    values='CSAT Score',
    fill_value=0
)

print("User-Item Matrix Sample:")
print(tabulate(user_item_matrix.head(), headers='keys', tablefmt='fancy_grid'), "\n")

# -----------------------------
# STEP 3: Compute Item Similarity (Cosine Similarity)
# -----------------------------
item_similarity = cosine_similarity(user_item_matrix.T)  # transpose to get items as rows
item_similarity_df = pd.DataFrame(item_similarity, index=user_item_matrix.columns, columns=user_item_matrix.columns)

print("Item Similarity Matrix Sample:")
print(tabulate(item_similarity_df.head(), headers='keys', tablefmt='fancy_grid'), "\n")

# -----------------------------
# STEP 4: Predict Ratings for All Users
# -----------------------------
predicted_ratings = user_item_matrix.dot(item_similarity_df) / item_similarity_df.sum(axis=1)
predicted_ratings = predicted_ratings.fillna(0)

print("Predicted Ratings Sample:")
print(tabulate(predicted_ratings.head(), headers='keys', tablefmt='fancy_grid'), "\n")

# -----------------------------
# STEP 5: Generate Top-N Recommendations
# -----------------------------
def top_n_recommendations(pred_ratings, n=3):
    recommendations = {}
    for user in pred_ratings.index:
        # Sort products by predicted score
        top_products = pred_ratings.loc[user].sort_values(ascending=False).head(n).index.tolist()
        recommendations[user] = top_products
    return recommendations

top_recs = top_n_recommendations(predicted_ratings, n=3)

# Convert to DataFrame for table display
top_recs_df = pd.DataFrame(list(top_recs.items()), columns=['UserID', 'Top_Products'])

print("Top 3 Recommended Products per User:")
print(tabulate(top_recs_df.head(10), headers='keys', tablefmt='fancy_grid'), "\n")

# -----------------------------
# STEP 6: Optional - Save Recommendations
# -----------------------------
top_recs_df.to_csv("week12_recommendations.csv", index=False)
print("Top recommendations saved to 'week12_recommendations.csv'")


